In [2]:
import pandas as pd
from pathlib import Path
import subprocess
import pandas as pd
import plotly.graph_objects as go

In [3]:
REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
REPO_ROOT

PosixPath('/Users/cyrilgabriele/Documents/Private/00_projects/finance/BaroqueTechnologies')

In [4]:
DATA_PATH = REPO_ROOT / "data" / "raw" / "crypto" / "BTCUSDT.csv"
BTC_PANEL_PATH = REPO_ROOT / "data/processed/strategies/crypto/btc/rsi_atr_stop_feature_panel.csv"

In [5]:
def run_command(command: list[str]) -> None: 
    result = subprocess.run(
        command, 
        cwd=REPO_ROOT,
        text=True, 
        capture_output=True
    )

    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if result.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {result.returncode}")


## Data Inestion

Belwo we ingest the data from Binance according to the config file in: 

- baroque/src/config/data/btc_data_config.yaml


In [6]:
run_command([
    "uv",
    "run",
    "python",
    "baroque/src/data/data_ingest/ingest_data.py",
    "--config",
    "baroque/src/config/data/btc_data_config.yaml",
])

Wrote 3228 rows to /Users/cyrilgabriele/Documents/Private/00_projects/finance/BaroqueTechnologies/data/raw/crypto/BTCUSDT.csv



In [7]:
btcusdt_df = pd.read_csv(DATA_PATH)
btcusdt_df

,date,timestamp,symbol,open,high,low,close
0,2017-08-17,2017-08-17T00:00:00Z,BTCUSDT,4261.48,4485.39,4200.74,4285.08
1,2017-08-18,2017-08-18T00:00:00Z,BTCUSDT,4285.08,4371.52,3938.77,4108.37
2,2017-08-19,2017-08-19T00:00:00Z,BTCUSDT,4108.37,4184.69,3850.00,4139.98
3,2017-08-20,2017-08-20T00:00:00Z,BTCUSDT,4120.98,4211.08,4032.62,4086.29
4,2017-08-21,2017-08-21T00:00:00Z,BTCUSDT,4069.13,4119.62,3911.79,4016.00
...,...,...,...,...,...,...,...
3223,2026-06-14,2026-06-14T00:00:00Z,BTCUSDT,64458.01,65800.00,63678.83,65746.45
3224,2026-06-15,2026-06-15T00:00:00Z,BTCUSDT,65746.45,67292.15,65354.00,66328.74
3225,2026-06-16,2026-06-16T00:00:00Z,BTCUSDT,66328.74,66992.00,65360.92,65675.01
3226,2026-06-17,2026-06-17T00:00:00Z,BTCUSDT,65675.02,66445.93,63915.77,64509.40


## Data Cleaning

Below we clean the data according to: 

- baroque/src/config/data/btc_data_config.yaml

This is the same config file as for the ingest script.

In [8]:
run_command([
    "uv",
    "run",
    "python",
    "baroque/src/data/data_engineering/data_cleaning.py",
    "--config",
    "baroque/src/config/data/btc_data_config.yaml",
])

Wrote 3228 rows across 1 symbols to /Users/cyrilgabriele/Documents/Private/00_projects/finance/BaroqueTechnologies/data/processed/btc_panel.csv (unbalanced panel).
BTCUSDT: 3228 -> 3228 rows, 2017-08-17 to 2026-06-18, dropped 0 rows.



## Feature Engineering 
Below we enhance the data with the needed features for the strategy

In [9]:
run_command([
    "uv",
    "run",
    "python",
    "baroque/src/data/data_engineering/strategies/mean_reversion/rsi_btc/rsi_btc.py",
    "--config",
    "baroque/src/config/strategies/crypto/btc/rsi_atr_stop_strategy.yaml",
])

Wrote 3228 rows to /Users/cyrilgabriele/Documents/Private/00_projects/finance/BaroqueTechnologies/data/processed/strategies/crypto/btc/rsi_atr_stop_feature_panel.csv



In [10]:
btc_panel_df = pd.read_csv(BTC_PANEL_PATH)
btc_panel_df.tail(50)


,date,timestamp,symbol,open,high,low,close,rsi_14,atr_14,oversold_armed,buy_signal,initial_stop_price
3178,2026-04-30,2026-04-30T00:00:00Z,BTCUSDT,75780.00,76669.14,75323.65,76346.57,55.659018,2266.416365,False,False,NaN
3179,2026-05-01,2026-05-01T00:00:00Z,BTCUSDT,76346.58,78914.12,76320.42,78231.13,61.054432,2289.793767,False,False,NaN
3180,2026-05-02,2026-05-02T00:00:00Z,BTCUSDT,78231.13,79199.48,78040.00,78686.85,62.250628,2209.057070,False,False,NaN
3181,2026-05-03,2026-05-03T00:00:00Z,BTCUSDT,78686.84,79447.00,78084.08,78568.57,61.720753,2148.618708,False,False,NaN
3182,2026-05-04,2026-05-04T00:00:00Z,BTCUSDT,78568.58,80776.99,78202.00,79861.01,65.205883,2179.073800,False,False,NaN
3183,2026-05-05,2026-05-05T00:00:00Z,BTCUSDT,79861.01,81791.48,79808.72,80905.52,67.760527,2165.051386,False,False,NaN
3184,2026-05-06,2026-05-06T00:00:00Z,BTCUSDT,80905.53,82850.00,80731.14,81447.01,69.030014,2161.752001,False,False,NaN
3185,2026-05-07,2026-05-07T00:00:00Z,BTCUSDT,81447.01,81708.32,79500.00,80006.00,62.029937,2165.078287,False,False,NaN
3186,2026-05-08,2026-05-08T00:00:00Z,BTCUSDT,80006.00,80500.00,79181.48,80193.17,62.560997,2104.609838,False,False,NaN
3187,2026-05-09,2026-05-09T00:00:00Z,BTCUSDT,80193.18,81080.00,80129.85,80678.40,63.967972,2022.148421,False,False,NaN


### Illustration via Interactive Plot

In [11]:
plot_df = btc_panel_df.copy()

plot_df["date"] = pd.to_datetime(plot_df["date"])

for column in ["open", "high", "low", "close", "rsi_14", "atr_14", "initial_stop_price"]:
    if column in plot_df.columns:
        plot_df[column] = pd.to_numeric(plot_df[column], errors="coerce")

for column in ["oversold_armed", "buy_signal"]:
    plot_df[column] = plot_df[column].astype(str).str.lower().isin(["true", "1", "yes"])

armed_df = plot_df[plot_df["oversold_armed"]].copy()
buy_df = plot_df[plot_df["buy_signal"]].copy()

In [12]:
fig = go.Figure()

fig.add_trace(
    go.Candlestick(
        x=plot_df["date"],
        open=plot_df["open"],
        high=plot_df["high"],
        low=plot_df["low"],
        close=plot_df["close"],
        name="BTCUSDT OHLC",
        increasing_line_color="#1f9d55",
        decreasing_line_color="#d64545",
    )
)

fig.add_trace(
    go.Scatter(
        x=armed_df["date"],
        y=armed_df["low"] * 0.995,
        mode="markers",
        name="Oversold armed",
        marker=dict(
            symbol="circle",
            size=7,
            color="#2563eb",
            opacity=0.65,
        ),
        customdata=armed_df[["close", "rsi_14", "atr_14"]],
        hovertemplate=(
            "<b>Oversold armed</b><br>"
            "Date: %{x|%Y-%m-%d}<br>"
            "Close: %{customdata[0]:.2f}<br>"
            "RSI(14): %{customdata[1]:.2f}<br>"
            "ATR(14): %{customdata[2]:.2f}"
            "<extra></extra>"
        ),
    )
)

fig.add_trace(
    go.Scatter(
        x=buy_df["date"],
        y=buy_df["close"],
        mode="markers",
        name="Buy signal",
        marker=dict(
            symbol="triangle-up",
            size=13,
            color="#f59e0b",
            line=dict(width=1, color="#111827"),
        ),
        customdata=buy_df[["close", "rsi_14", "atr_14", "initial_stop_price"]],
        hovertemplate=(
            "<b>Buy signal</b><br>"
            "Date: %{x|%Y-%m-%d}<br>"
            "Buy price / close: %{customdata[0]:.2f}<br>"
            "RSI(14): %{customdata[1]:.2f}<br>"
            "ATR(14): %{customdata[2]:.2f}<br>"
            "Initial stop: %{customdata[3]:.2f}"
            "<extra></extra>"
        ),
    )
)

fig.add_trace(
    go.Scatter(
        x=buy_df["date"],
        y=buy_df["initial_stop_price"],
        mode="markers",
        name="Initial stop on buy",
        marker=dict(
            symbol="x",
            size=10,
            color="#dc2626",
        ),
        customdata=buy_df[["close", "initial_stop_price"]],
        hovertemplate=(
            "<b>Initial stop</b><br>"
            "Date: %{x|%Y-%m-%d}<br>"
            "Entry close: %{customdata[0]:.2f}<br>"
            "Initial stop: %{customdata[1]:.2f}"
            "<extra></extra>"
        ),
    )
)

fig.update_layout(
    title="BTCUSDT RSI/ATR Entry Signals",
    xaxis_title="Date",
    yaxis_title="Price",
    template="plotly_white",
    height=750,
    hovermode="x unified",
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="left",
        x=0,
    ),
)

fig.update_xaxes(
    rangeslider_visible=True,
    rangeselector=dict(
        buttons=[
            dict(count=6, label="6M", step="month", stepmode="backward"),
            dict(count=1, label="1Y", step="year", stepmode="backward"),
            dict(count=2, label="2Y", step="year", stepmode="backward"),
            dict(step="all", label="All"),
        ]
    ),
)

fig.show()